# SpecDist — Kaggle Quickstart

**Free GPU: 30 h/week · T4/P100 · 12 h sessions (vs ~90 min on free Colab)**

| Cell | What it does | Time |
|------|-------------|------|
| 1. Setup | Clone repo · install deps · auth | ~3 min |
| 2. Run | Full pipeline → `/kaggle/working/` | ~4 h |
| 3. Save | Download checkpoint as Kaggle dataset | ~5 min |
| 4. Resume | Attach saved checkpoint dataset · continue training | ~1 min + train |

---

### Before you start
1. **GPU runtime**: Settings → Accelerator → GPU T4 × 1
2. **Kaggle secrets** (Add-ons → Secrets):
   - `WANDB_API_KEY` — https://wandb.ai/authorize
   - `HF_TOKEN` — https://huggingface.co/settings/tokens
3. **Internet enabled**: Settings → Internet → On

> **Storage note:** `/kaggle/working/` persists within a session but is lost on restart.
> Save checkpoints using Cell 3 after each session (saves as a Kaggle dataset output).

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 1 — Setup + Auth
# ─────────────────────────────────────────────────────────────────────────────
import os, subprocess, sys

REPO_URL   = "https://github.com/Rmuk655/Distill-Spec-Research.git"
REPO_DIR   = "/kaggle/working/Distill-Spec-Research"
GBV_DIR    = f"{REPO_DIR}/gbv-research"
CKPT_ROOT  = "/kaggle/working/specdist/checkpoints"
HF_HOME    = "/kaggle/working/hf_cache"
os.makedirs(CKPT_ROOT, exist_ok=True)
os.makedirs(HF_HOME,   exist_ok=True)

# ── Clone / update repo ──────────────────────────────────────────────────────
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    print("✓ Repo cloned")
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
    print("✓ Repo updated")

os.chdir(GBV_DIR)

# ── Install deps ─────────────────────────────────────────────────────────────
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "-r", "requirements.txt", "bitsandbytes", "accelerate"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "flash-attn", "--no-build-isolation"], check=False)  # optional speedup
print("✓ Dependencies installed")

# ── Auth: read from Kaggle Secrets ────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()

    wandb_key = _secrets.get_secret("WANDB_API_KEY")
    os.environ["WANDB_API_KEY"] = wandb_key
    import wandb; wandb.login(key=wandb_key, relogin=True)
    print("✓ W&B authenticated")
except Exception as e:
    print(f"⚠ W&B key not found ({e}) — logging offline")
    os.environ["WANDB_MODE"] = "offline"

try:
    hf_token = _secrets.get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
    print("✓ HF token set")
except Exception:
    print("ℹ HF_TOKEN not found — OK for public models")

# ── GPU check ─────────────────────────────────────────────────────────────────
import torch
os.environ["HF_HOME"] = HF_HOME
os.environ["TRANSFORMERS_OFFLINE"] = "0"
os.environ["HF_HUB_OFFLINE"] = "0"
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info(0)
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}  {free/1024**3:.1f}/{total/1024**3:.1f} GB")
else:
    print("⚠ No GPU — enable GPU in Settings → Accelerator")

print(f"\n✓ Setup done. Checkpoints → {CKPT_ROOT}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 2 — Run pipeline
#
# CONFIG options:
#   colab  → T4/P100 (15 GB): 8B teacher in 4-bit NF4  ← use this on Kaggle
#   server → A100 (40 GB):    8B teacher in bfloat16
#
# Set SMOKE=True for a ~45-min code-path check before the overnight run.
# ─────────────────────────────────────────────────────────────────────────────
import os, subprocess, sys

GBV_DIR   = "/kaggle/working/Distill-Spec-Research/gbv-research"
CKPT_ROOT = "/kaggle/working/specdist/checkpoints"

CONFIG = "colab"    # T4/P100 Kaggle GPU — same as free Colab
SMOKE  = False      # True = quick test (~45 min)
LOSSES = None       # None = all 6 losses; or e.g. "kl,ebe"

cmd = [
    sys.executable, "orchestration/pipeline.py",
    "--config",    CONFIG,
    "--ckpt_root", CKPT_ROOT,
    "--yes",
]
if SMOKE:  cmd.append("--smoke")
if LOSSES: cmd += ["--losses", LOSSES]

print(f"Running: {' '.join(cmd)}")
result = subprocess.run(cmd, cwd=GBV_DIR)

if result.returncode == 0:
    print("\n✓ Pipeline complete! Run Cell 3 to save checkpoints.")
else:
    print(f"\n✗ Exited with code {result.returncode}")
    print("  Fix the error above, then re-run — the pipeline resumes from last checkpoint.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 3 — Save checkpoints as Kaggle output
#
# Kaggle kernels produce output files accessible from the Output tab.
# Anything written to /kaggle/working/ is saved as notebook output
# and can be attached as a dataset to future notebooks for resume.
#
# After the session ends, go to:  Your notebook → Output → Download
# ─────────────────────────────────────────────────────────────────────────────
import os, shutil, json

CKPT_ROOT  = "/kaggle/working/specdist/checkpoints"
GBV_DIR    = "/kaggle/working/Distill-Spec-Research/gbv-research"

# Save a status summary alongside checkpoints
status = {
    "checkpoints": os.listdir(CKPT_ROOT) if os.path.isdir(CKPT_ROOT) else [],
    "results_db_bytes": os.path.getsize(f"{GBV_DIR}/db/results.db")
                        if os.path.exists(f"{GBV_DIR}/db/results.db") else 0,
}
with open("/kaggle/working/specdist/run_status.json", "w") as f:
    json.dump(status, f, indent=2)

# Copy results.db into /kaggle/working/specdist so it's included in output
db_src = f"{GBV_DIR}/db/results.db"
db_dst = "/kaggle/working/specdist/results.db"
if os.path.exists(db_src):
    shutil.copy(db_src, db_dst)
    print(f"✓ results.db saved ({os.path.getsize(db_dst)} bytes)")

print(f"\n✓ Output saved to /kaggle/working/specdist/")
print(f"  Checkpoints : {status['checkpoints']}")
print(f"\n  ↓ Download: Your notebook → Output tab → Download button")
print(f"  ↓ Resume:   Attach the downloaded dataset, run Cell 4")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 4 — Resume from a previously saved checkpoint dataset
#
# Prerequisites:
#   1. Add the saved checkpoint dataset to this notebook:
#      Notebook → Data → + Add Data → Your Datasets → select specdist-checkpoints
#   2. The dataset will be mounted at /kaggle/input/specdist-checkpoints/
#   3. Run Cells 1 and 4 (skip Cell 2 for now)
# ─────────────────────────────────────────────────────────────────────────────
import os, shutil, subprocess, sys

GBV_DIR       = "/kaggle/working/Distill-Spec-Research/gbv-research"
CKPT_ROOT     = "/kaggle/working/specdist/checkpoints"
DATASET_INPUT = "/kaggle/input/specdist-checkpoints"  # adjust dataset name if different

# ── Copy checkpoints from dataset input → working directory ──────────────────
if os.path.isdir(DATASET_INPUT):
    os.makedirs(CKPT_ROOT, exist_ok=True)
    for item in os.listdir(DATASET_INPUT):
        src = os.path.join(DATASET_INPUT, item)
        dst = os.path.join(CKPT_ROOT, item)
        if not os.path.exists(dst):
            if os.path.isdir(src):
                shutil.copytree(src, dst)
            else:
                shutil.copy(src, dst)
    print(f"✓ Checkpoints restored from {DATASET_INPUT}")
    print(f"  Contents: {os.listdir(CKPT_ROOT)}")
else:
    print(f"ℹ No checkpoint dataset attached — starting from scratch")
    print(f"  To attach: Notebook → Data → + Add Data → Your Datasets")

# ── Resume pipeline ───────────────────────────────────────────────────────────
CONFIG = "colab"
result = subprocess.run([
    sys.executable, "orchestration/pipeline.py",
    "--config",    CONFIG,
    "--ckpt_root", CKPT_ROOT,
    "--yes",
], cwd=GBV_DIR)

print(f"\nPipeline exited with code {result.returncode}")
print("Run Cell 3 to save the updated checkpoints.")